# 04 · Phase 0 clutter pilot (E0.7): primary clutter level and Phase 1 step budget  (~0.55 GPU-h)

**Accelerator:** GPU T4 ×2 · **Internet:** on · **Inputs:** none (N\* = 32 is read from `configs/prereg.yaml`)

**Why:** NB02 showed that A1 (GDN) solves N = N\* = 32 pairs at 99% *without* clutter (it breaks the plateau at
~1,900 steps), but with F = 512 clutter tokens it never leaves chance in 6,000 steps (A1 and A2 at 0.04%). With
every arm at zero, Phase 1 could not show whether Engram helps. This pilot finds the clutter level where A1 still
learns, and gives the Phase 1 step budget from that run's curve.

**Runs (A1 only, N = N\*, P = 64, placement a, 6,000 steps, val recall every 250 steps):**

| F (clutter tokens) | filler loss λ | purpose |
|---|---|---|
| 64 | 1 | backup level, in case 128 already fails |
| 128 | 1 | design level |
| 256 | 1 | design level |
| 512 | 0 | is the filler loss what blocks learning at 512? |

**Pre-registered rule (PREREG.md):** primary F = the largest of {256, 128} where A1's max val recall ≥ 0.5;
`phase1_steps` = the 98% rule on that run's curve. If neither learns, the notebook **stops** and you decide
(the F = 64 and λ = 0 runs inform that decision).

**Outputs:** the pilot table and curves, `results/prereg.proposed.yaml` (N_star, phase1_steps, primary F) and the
Phase 1 GPU-hour estimate.

### Logs and downloads (created automatically while this notebook runs)

Everything appears in this notebook's **Output** tab (`/kaggle/working`) and can be downloaded directly:

| File | What it contains |
|---|---|
| `run.log` | **The whole run**: everything this notebook prints, live, with elapsed seconds per line: every command, every training step, every warning and the full traceback of any failed cell |
| `results/_logs/<group>/<run_id>.log` | One log per training run (the full output of that run, both GPUs) |
| `04_phase0_clutter_pilot_results.zip` | Rebuilt after every cell: `results/` (result JSONs + all the logs above, no checkpoints). **Download this one** |

To hand the results back: download `04_phase0_clutter_pilot_results.zip` and run
`unzip -n 04_phase0_clutter_pilot_results.zip -d <your Engram repo>`. If the session dies, `run.log` and the per-run logs
still hold everything up to that moment. The code itself is cloned to `/tmp` and is **not** part of the output.

In [ ]:
# ===================== SETTINGS: edit this cell, then "Run All" (or "Save Version -> Save & Run All") =====================
REPO_URL     = "https://github.com/Shiveshrane/Engram.git"   # None -> copy the code from CODE_DATASET instead
REPO_REF     = "main"           # pin the PREREG commit hash for Phase 1 / 2 runs
CODE_DATASET = "/kaggle/input/engramdelta-src"   # Kaggle Dataset with the repo (used when REPO_URL is None)
HF_REPO      = None             # e.g. "your-hf-user/engramdelta-runs" (private dataset repo; needs Kaggle Secret HF_TOKEN)
GDN_BACKEND  = "torch"          # "torch" (pure PyTorch, always works) or "fla" (only if NB00 says PASS)
FLA_SPEC     = "fla-core==0.5.2"                 # FLA's kernel package (fla.ops); installed with --no-deps
PREV_OUTPUTS = []               # outputs of earlier sessions attached as inputs; globs work, e.g. ["/kaggle/input/*"]
NCCL_ENV     = {}               # set to {"NCCL_P2P_DISABLE": "1", "NCCL_IB_DISABLE": "1"} if NB00's first NCCL test hung

In [ ]:
# ===================== Session setup (no edits needed) =====================
import os, sys, subprocess, shutil, pathlib, json, glob, time, datetime, traceback, zipfile
NEEDS_GPU, INSTALL_FLA = True, False
NB_NAME = "04_phase0_clutter_pilot"
ON_KAGGLE = os.path.isdir("/kaggle/working")
WORK = pathlib.Path("/kaggle/working") if ON_KAGGLE else pathlib.Path.cwd()
# On Kaggle the code is cloned to /tmp, which is NOT saved, so the notebook output holds only what this session
# creates: results/, figures/, logs/, ckpt/ (unfinished runs only) and the results zip. results/ and figures/ are
# symlinked from the repo into /kaggle/working. Locally (notebook opened from the repo's notebooks/ folder) the
# repo is the parent directory.
REPO = pathlib.Path("/tmp/engramdelta") if ON_KAGGLE else (WORK.parent if WORK.name == "notebooks" else WORK)

# 0. run.log: everything this notebook prints (every command, every training step, every error traceback) is also
#    written, live, to run.log at the top of the notebook output, with elapsed seconds per line. Each training run
#    also gets its own log, results/_logs/<group>/<run_id>.log. After every cell, results/ (result JSONs + all logs,
#    no checkpoints) is zipped into <notebook>_results.zip. All three are in the Output tab, ready to download.
class _Tee:
    _engram_tee = True
    def __init__(self, stream, f, t0):
        self._s, self._f, self._t0, self._bol = stream, f, t0, True
    def write(self, s):
        out = []
        for part in s.splitlines(keepends=True):
            if self._bol:
                out.append(f"[{time.time() - self._t0:9.1f}s] ")
            out.append(part)
            self._bol = part.endswith("\n")
        self._f.write("".join(out)); self._f.flush()
        return self._s.write(s)
    def flush(self):
        self._s.flush(); self._f.flush()
    def __getattr__(self, k):
        return getattr(self._s, k)

if not getattr(sys.stdout, "_engram_tee", False):      # setup cell re-run: keep the existing log
    _STAMP = datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ')
    LOG_PATH = WORK / "run.log"
    _LOG = open(LOG_PATH, "a", buffering=1)
    _LOG.write(f"===== {NB_NAME} | started {_STAMP} (UTC) =====\n")
    _T0 = time.time()
    sys.stdout, sys.stderr = _Tee(sys.stdout, _LOG, _T0), _Tee(sys.stderr, _LOG, _T0)
ZIP_PATH = WORK / f"{NB_NAME}_results.zip"

def bundle():
    """Copy the session log into results/_logs/notebooks/ and zip results/ into ZIP_PATH (no checkpoints)."""
    try:
        res = REPO / "results"
        if not res.is_dir():
            return
        dst = res / "_logs" / "notebooks"; dst.mkdir(parents=True, exist_ok=True)
        _LOG.flush(); shutil.copy2(LOG_PATH, dst / f"{NB_NAME}__{_STAMP}.log")
        tmp = ZIP_PATH.with_name(ZIP_PATH.name + ".tmp")
        with zipfile.ZipFile(tmp, "w", zipfile.ZIP_DEFLATED) as z:
            for root, _, files in os.walk(res):
                for fn in files:
                    fp = os.path.join(root, fn); z.write(fp, os.path.relpath(fp, REPO))
        os.replace(tmp, ZIP_PATH)
    except Exception as e:      # never let logging break the experiment
        sys.__stdout__.write(f"!! results zip failed: {e}\n")

_ip = get_ipython()
for _ev in ("pre_run_cell", "post_run_cell"):
    for _cb in list(_ip.events.callbacks[_ev]):
        if getattr(_cb, "__name__", "").startswith("_engram_"):
            _ip.events.unregister(_ev, _cb)
def _engram_pre(info=None):
    src = (getattr(info, "raw_cell", "") or "").strip().splitlines()
    _LOG.write(f"\n===== cell start {time.strftime('%H:%M:%S')} | {src[0][:100] if src else ''}\n")
def _engram_post(result=None):
    err = getattr(result, "error_in_exec", None) or getattr(result, "error_before_exec", None)
    if err is not None:
        _LOG.write("!! CELL ERROR\n" + "".join(traceback.format_exception(type(err), err, err.__traceback__)))
    bundle()
_ip.events.register("pre_run_cell", _engram_pre)
_ip.events.register("post_run_cell", _engram_post)
print("=" * 100)
print(f"LOGS  run.log (this whole notebook, live)  -> {LOG_PATH}")
print(f"      one log per training run             -> {REPO / 'results' / '_logs'}/<group>/<run_id>.log")
print(f"      everything zipped after every cell   -> {ZIP_PATH}")
print("      Download from the notebook's Output tab (run.log and the zip are at the top level).")
print("=" * 100)

def run(cmd, check=True, env=None, timeout=None, show=None):
    """Run a shell command in the repo and stream its output into this notebook."""
    print("$", show or cmd, flush=True)
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                         env={**os.environ, **(env or {})}, cwd=str(REPO) if REPO.exists() else str(WORK))
    t0 = time.time()
    try:
        for line in p.stdout:
            print(line, end="", flush=True)
            if timeout and time.time() - t0 > timeout:
                raise subprocess.TimeoutExpired(cmd, timeout)
        rc = p.wait(timeout=timeout)
    except subprocess.TimeoutExpired:
        p.kill(); print(f"!! timed out after {timeout}s"); rc = -9
    if check and rc != 0:
        raise RuntimeError(f"exit code {rc}: {show or cmd}")
    return rc

def secret(name):
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        return os.environ.get(name)

# 1. code
if not REPO.exists():
    if REPO_URL:
        url, tok = REPO_URL, secret("GITHUB_TOKEN")
        if tok and url.startswith("https://"):
            url = url.replace("https://", f"https://{tok}@")
        run(f"git clone -q {url} {REPO}", show=f"git clone {REPO_URL}")
        run(f"git -C {REPO} checkout -q {REPO_REF}")
    else:
        shutil.copytree(CODE_DATASET, REPO)
if ON_KAGGLE:
    for d in ("results", "figures"):
        out, link = WORK / d, REPO / d
        out.mkdir(parents=True, exist_ok=True)
        if not link.is_symlink():
            if link.exists():
                shutil.copytree(link, out, dirs_exist_ok=True); shutil.rmtree(link)
            link.symlink_to(out, target_is_directory=True)
os.chdir(REPO); sys.path.insert(0, str(REPO))
try:
    commit = subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True, check=True).stdout.strip()
except Exception:
    commit = open("GIT_COMMIT").read().strip() if os.path.exists("GIT_COMMIT") else "unknown"
os.environ["GIT_COMMIT"] = commit
print("code commit:", commit)

# 2. packages (never let pip replace Kaggle's torch)
run("pip install -q --no-deps einops sympy")
if INSTALL_FLA or GDN_BACKEND == "fla":
    run(f"pip install -q --no-deps {FLA_SPEC}", check=False)

# 3. environment for every launched run
os.environ["ENGRAMDELTA_GDN_BACKEND"] = GDN_BACKEND
os.environ["ENGRAMDELTA_CKPT_DIR"] = str(WORK / "ckpt")
os.environ.update(NCCL_ENV)
if HF_REPO and secret("HF_TOKEN"):
    os.environ["HF_REPO"], os.environ["HF_TOKEN"] = HF_REPO, secret("HF_TOKEN")
    print("HF Hub sync: ON ->", HF_REPO)
else:
    print("HF Hub sync: OFF (results persist via this notebook's output only)")

# 4. restore results / checkpoints from earlier sessions (never overwrites existing files)
def restore(src, dst):
    if not os.path.isdir(src):
        return 0
    n = 0
    for root, _, files in os.walk(src):
        for f in files:
            s = os.path.join(root, f); d = os.path.join(dst, os.path.relpath(s, src))
            if not os.path.exists(d):
                os.makedirs(os.path.dirname(d), exist_ok=True); shutil.copy2(s, d); n += 1
    return n
if ON_KAGGLE and os.path.isdir("/kaggle/input"):
    print("attached inputs:", sorted(glob.glob("/kaggle/input/*")) or "none")
def find_dirs(prev, name):
    """Folders called `name` up to 4 levels inside an attached output (e.g. <prev>/results, <prev>/engramdelta/results)."""
    c = [p for k in range(5) for p in glob.glob(os.path.join(prev, *(["*"] * k), name))]
    return [d for d in dict.fromkeys(c) if os.path.isdir(d) and "/_logs/" not in d]
_prevs = [p for e in PREV_OUTPUTS for p in (sorted(glob.glob(e)) or [e])]     # entries may be globs, e.g. "/kaggle/input/*"
for prev in _prevs:
    if not os.path.isdir(prev):
        print(f"!! PREV_OUTPUTS entry not found: {prev}  (check the 'attached inputs' list above)"); continue
    dirs = find_dirs(prev, "results")
    n1 = sum(restore(d, str(REPO / "results")) for d in dirs)
    # checkpoints: only for runs that have no result yet (a finished run never resumes, so its checkpoint is skipped)
    done_ids = {os.path.splitext(f)[0] for _, _, fs in os.walk(REPO / "results") for f in fs if f.endswith(".json")}
    n2 = 0
    for ck in [c for d in find_dirs(prev, "ckpt") for c in glob.glob(os.path.join(d, "*"))]:
        if os.path.basename(ck) not in done_ids:
            n2 += restore(ck, str(WORK / "ckpt" / os.path.basename(ck)))
    print(f"restored from {prev}: {n1} result files (from {dirs}), {n2} checkpoint files of unfinished runs")
if os.environ.get("HF_REPO"):
    run("python -m engramdelta.scripts.sync pull results", check=False)

# 5. hardware
if NEEDS_GPU:
    run("nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv", check=False)
import torch
print("torch", torch.__version__, "| CUDA", torch.version.cuda, "| GPUs:", torch.cuda.device_count())
NPROC = max(1, torch.cuda.device_count())
if NEEDS_GPU and NPROC < 2:
    print("!! WARNING: fewer than 2 GPUs. Set Accelerator = 'GPU T4 x2' in the notebook settings.")

## E0.7 · clutter pilot runs

In [ ]:
GROUPS = ['E0.7']     # run in this order (later groups may need earlier results, e.g. LRs from E1.0a)
SESSION_HOURS = 1.0   # leave margin below Kaggle's 12 h session limit
EXTRA = ""
t_start = time.time()
for g in GROUPS:
    left = SESSION_HOURS - (time.time() - t_start) / 3600
    if left < 0.15:
        print("session budget used up; rerun this notebook to continue"); break
    run(f"python -m engramdelta.scripts.make_manifest --group {g} --out manifests {EXTRA}")
    run(f"python -m engramdelta.scripts.budget manifests/{g}.yaml --nproc {NPROC}", check=False)
    rc = run(f"python -m engramdelta.scripts.sweep manifests/{g}.yaml --nproc {NPROC} --session-hours {left:.2f}", check=False)
    if rc != 0:
        print(f"!! sweep for {g} reported failures or stopped early (exit {rc}). Check results/_failures before continuing.")

## Result: which clutter level does A1 still learn?

In [ ]:
import yaml
from engramdelta import analysis
pr = yaml.safe_load(open("configs/prereg.yaml"))
N_STAR = int(pr["N_star"])
rows = analysis.clutter_pilot("results", N=N_STAR)
print(f"A1 at N={N_STAR}, P=64, placement a, 6000 steps   (chance = 1/2048 = 0.0005)")
print(f"{'F':>4} {'lambda':>6} {'max val recall':>15} {'final':>7} {'learned':>8} {'first step >= 98% of max':>25} {'min':>5}")
for x in rows:
    print(f"{x['F']:>4} {x['lam']:>6g} {x['max_val_recall']:>15.3f} {x['final_val_recall']:>7.3f} {str(x['learned']):>8} "
          f"{x['first_step_at_98pct'] if x['learned'] else '-':>25} {x['train_seconds'] / 60:>5.1f}")
for x in rows:
    print(f"  F={x['F']} lambda={x['lam']:g} curve:", [(st, round(v, 3)) for st, v in x["curve"]])

import matplotlib.pyplot as plt
os.makedirs("figures", exist_ok=True)
fig, ax = plt.subplots(figsize=(7, 4))
for x in rows:
    st, v = zip(*x["curve"])
    ax.plot(st, v, linewidth=2, label=f"F={x['F']}, lambda={x['lam']:g}")
ax.set_xlabel("step"); ax.set_ylabel("val recall"); ax.set_ylim(0, 1.02)
ax.set_title(f"E0.7 clutter pilot: A1 at N={N_STAR}"); ax.legend(frameon=False)
for side in ("top", "right"):
    ax.spines[side].set_visible(False)
fig.savefig("figures/E0.7_clutter_pilot.png", dpi=150, bbox_inches="tight"); plt.show()

if len(rows) < 4:
    print("!! only", len(rows), "of 4 pilot runs finished (session time?). Rerun with this notebook's output in PREV_OUTPUTS.")
choice = analysis.primary_f_rule(rows)
if choice is None:
    raise RuntimeError("STOP: A1 did not reach 0.5 val recall at F=256 or F=128. Bring this table back for a decision "
                       "(see the F=64 and lambda=0 rows).")
PRIMARY_F, PHASE1_STEPS = choice["F"], choice["proposal"]
print(f"\n>> primary F = {PRIMARY_F}   (largest of 256, 128 where A1 learns)")
print(f">> phase1_steps = {PHASE1_STEPS}   (first step >= 98% of max at F={PRIMARY_F}: {choice['first_step_at_98pct']}, rounded up to 500)")

## Proposed pre-registered values and the Phase 1 budget (from measured throughput)

In [ ]:
prop = dict(pr, phase1_steps=PHASE1_STEPS, primary_cell=dict(pr["primary_cell"], F=PRIMARY_F))
yaml.safe_dump(prop, open("configs/prereg.proposed.yaml", "w"), sort_keys=False)
shutil.copy2("configs/prereg.proposed.yaml", "results/prereg.proposed.yaml")   # also in the output / zip
print(open("configs/prereg.proposed.yaml").read())
tmp = dict(prop, lr={"A1": 1e-3, "A2": 1e-3, "T1": 1e-3})     # LRs are not known yet; they do not change run time
yaml.safe_dump(tmp, open("/tmp/prereg_budget.yaml", "w"))
run("python -m engramdelta.scripts.make_manifest --group all-phase1 --prereg /tmp/prereg_budget.yaml --out /tmp/manifests_p1")
run(f"python -m engramdelta.scripts.budget '/tmp/manifests_p1/*.yaml' --nproc {NPROC} --quota-hours 13", check=False)

## Your turn (no GPU needed)
1. Download `04_phase0_clutter_pilot_results.zip` and give it to Claude, or read the two `>>` lines above.
2. If the budget says OVER BUDGET, decide on cuts **before** committing.
3. Copy `phase1_steps` and `primary_cell.F` from `results/prereg.proposed.yaml` into `configs/prereg.yaml`, fill the
   PREREG.md "Phase 0 outcomes" table, commit, push, and put the commit hash in PREREG.md and as `REPO_REF` in
   notebooks 10–14. Then run NB10.

In [ ]:
# ===================== End of session: persist =====================
if os.environ.get("HF_REPO"):
    run("python -m engramdelta.scripts.sync push results", check=False)
n = sum(len(f) for _, _, f in os.walk(REPO / "results"))
print(f"{n} files under results/. They are part of this notebook's output: Save Version keeps them.")
print("Next session: attach this notebook's output as an input and add its path to PREV_OUTPUTS (or use HF_REPO).")
bundle()
_mb = ZIP_PATH.stat().st_size / 1e6 if ZIP_PATH.exists() else 0.0
print(f"\nTo send the results back: Output tab -> download {ZIP_PATH.name} ({_mb:.1f} MB) "
      "and unzip it into the repo folder:")
print(f"    unzip -n {ZIP_PATH.name} -d <your Engram repo>     # -n: never overwrite existing files")
print("It holds results/ (result JSONs), results/_logs/<group>/<run_id>.log (each run's training log)")
print(f"and results/_logs/notebooks/{NB_NAME}__{_STAMP}.log (a copy of run.log).")
print(f"run.log alone (this notebook's full printed output) is also in the Output tab: {LOG_PATH}")